# MemeTector v4 — A100 최적화 버전

## v3 → v4 핵심 개선사항
| 항목 | v3 | v4 | 예상 효과 |
|------|-----|-----|----------|
| 백본 | CLIP ViT-L/14 (224px) | **CLIP ViT-L/14@336px** | +1~2% |
| 데이터 | MEME 500 / NOT_MEME 500 (샘플링) | **ALL 502 / 736** (전체 사용) | +1~2% |
| 학습 전략 | 단일 split | **5-Fold CV + 앙상블** | +2~4% |
| 융합 방식 | concat + cosine | **Gated Fusion** (학습 가능한 가중합) | +0.5~1% |
| Augmentation | MixUp | **CutMix + MixUp + RandomErasing** | +0.5~1% |
| 정밀도 | float16 | **bfloat16** (A100 네이티브, 오버플로우 없음) | 안정성 ↑ |
| 배치 크기 | 32 | **64** (A100 메모리 최대 활용) | 학습 안정성 ↑ |
| TTA | 10-view | **12-view** | 소폭 향상 |

**핵심 문제 분석 (v3):** Val 91.4% vs Test 86.0% → **5.4% 갭** = 과적합/분산 문제  
**v4 해결책:** 5-Fold 앙상블 + 전체 데이터 활용 + 강화된 정규화

**준비:** 런타임 → **A100 GPU** 선택 / 구글 드라이브에 `Meme_26SP/content/` 폴더 확인

## 메모리 최적화 수정 사항 (OOM 방지)
| 수정 항목 | 내용 |
|-----------|------|
| `PYTORCH_ALLOC_CONF` | `expandable_segments:True` — 단편화 방지 |
| EasyOCR | `gpu=False` — OCR/CLIP 동시 GPU 점유 방지 |
| `zero_grad` | `set_to_none=True` — 그래디언트 메모리 즉시 해제 |
| `cutmix_data` | `pv.clone()` 제거 — 배치 2배 점유 해결 |
| Phase 1→2 전환 | `del opt1, sch1` 후 `empty_cache()` |
| Phase 2 루프 | 배치 텐서 매 스텝 해제 + 주기적 `empty_cache()` |
| Fold 마무리 | `del opt2, sch2, model` + `gc.collect()` |
| TTA flush | GPU 텐서 즉시 해제 |


In [1]:
# ── 패키지 설치 ────────────────────────────────────────────────
!pip install -q transformers accelerate scikit-learn tqdm Pillow easyocr opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 34.5 MB/s eta 0:00:00


In [2]:
# ── 임포트 ──────────────────────────────────────────────────────
import os, json, math, random
from pathlib import Path
from collections import Counter

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.amp import autocast
import torchvision.transforms as T

from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import easyocr
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  GPU 없음 — A100 런타임으로 변경 필요')

PyTorch: 2.10.0+cu128
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [4]:
# ── Google Drive 마운트 ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/Meme_26SP/content/'
OUT_DIR  = '/content/drive/MyDrive/Meme_26SP/memetector_v4'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'출력 폴더: {OUT_DIR}')

Mounted at /content/drive
출력 폴더: /content/drive/MyDrive/Meme_26SP/memetector_v4


In [5]:
# ── 전체 데이터 로드 (v3 대비 +238개) ─────────────────────────
# v3: MEME 500 / NOT_MEME 500 (샘플링으로 일부 손실)
# v4: 가용한 모든 데이터 사용 → 더 많은 학습 정보

SEED = 42
data_dir = Path(DATA_DIR)
exts     = {'.jpg', '.jpeg', '.png', '.webp'}

def get_images(d):
    return sorted([p for p in Path(d).iterdir() if p.suffix.lower() in exts])

meme_all     = get_images(data_dir / 'MEME')            # 502
notmeme_all  = get_images(data_dir / 'NOT_MEME')        # 602
untitled_all = get_images(data_dir / 'untitled folder') # 134

# 전체 사용 (v3: 각 500개 샘플링 → v4: 전부)
meme_paths    = meme_all
notmeme_paths = notmeme_all + untitled_all  # 736개

print(f'MEME:     {len(meme_paths)}개  (v3: 500개 → +{len(meme_paths)-500}개)')
print(f'NOT_MEME: {len(notmeme_paths)}개  (v3: 500개 → +{len(notmeme_paths)-500}개)')
print(f'총:       {len(meme_paths)+len(notmeme_paths)}개  (v3: 1000개 → +{len(meme_paths)+len(notmeme_paths)-1000}개)')

MEME:     502개  (v3: 500개 → +2개)
NOT_MEME: 736개  (v3: 500개 → +236개)
총:       1238개  (v3: 1000개 → +238개)


In [6]:
# ── Visual Part + OCR 처리 ─────────────────────────────────────
VISUAL_PART_DIR = '/content/visual_parts_v4'
os.makedirs(VISUAL_PART_DIR, exist_ok=True)

def merge_boxes(boxes, gap=50):
    if not boxes: return []
    boxes = sorted(boxes, key=lambda b: (b[1], b[0]))
    merged = [list(boxes[0])]
    for x1, y1, x2, y2 in boxes[1:]:
        mx1, my1, mx2, my2 = merged[-1]
        if min(y2, my2) - max(y1, my1) > 0 and min(abs(x1-mx2), abs(mx1-x2)) < gap:
            merged[-1] = [min(x1,mx1), min(y1,my1), max(x2,mx2), max(y2,my2)]
        else:
            merged.append([x1, y1, x2, y2])
    return [tuple(b) for b in merged]

def create_visual_part(img_path, ocr_reader, save_dir,
                        conf_thresh=0.15, pad=15, inpaint_radius=7):
    save_path = Path(save_dir) / (Path(img_path).stem + '.jpg')
    if save_path.exists(): return save_path
    try:
        img = Image.open(img_path).convert('RGB')
        arr = np.array(img)
        dets = ocr_reader.readtext(arr, detail=1, paragraph=False)
        mask = np.zeros(arr.shape[:2], np.uint8)
        raw_boxes = []
        for (bbox, txt, conf) in dets:
            if conf < conf_thresh or len(txt.strip()) < 2: continue
            xs = [int(p[0]) for p in bbox]; ys = [int(p[1]) for p in bbox]
            raw_boxes.append((max(0,min(xs)-pad), max(0,min(ys)-pad),
                               min(img.width,max(xs)+pad), min(img.height,max(ys)+pad)))
        for (x1,y1,x2,y2) in merge_boxes(raw_boxes):
            mask[y1:y2, x1:x2] = 255
        if mask.sum() > 0:
            result = cv2.inpaint(arr, mask, inpaintRadius=inpaint_radius, flags=cv2.INPAINT_TELEA)
            Image.fromarray(result).save(save_path, quality=92)
        else:
            img.save(save_path, quality=92)
    except:
        Image.open(img_path).convert('RGB').save(save_path, quality=92)
    return save_path

def extract_ocr_text(img_path, ocr_reader, conf_thresh=0.25, max_chars=200):
    try:
        arr  = np.array(Image.open(img_path).convert('RGB'))
        dets = ocr_reader.readtext(arr, detail=1, paragraph=False)
        txts = [t.strip() for (_, t, c) in dets if c >= conf_thresh and len(t.strip()) >= 2]
        combined = ' '.join(txts)
        return combined[:max_chars] if combined else ''
    except:
        return ''

print('EasyOCR 초기화 중...')
ocr_reader = easyocr.Reader(['en', 'ko'], gpu=False, verbose=False)  # CPU전용: OCR과 CLIP 동시 GPU 점유 방지
print('✅ 완료')

print(f'\nMEME {len(meme_paths)}개 처리 중 (OCR + Visual Part)...')
meme_texts, vp_map = {}, {}
for p in tqdm(meme_paths, desc='MEME'):
    meme_texts[str(p)] = extract_ocr_text(p, ocr_reader)
    vp_map[str(p)]     = create_visual_part(p, ocr_reader, VISUAL_PART_DIR)

print(f'\nNOT_MEME {len(notmeme_paths)}개 OCR 중...')
notmeme_texts = {}
for p in tqdm(notmeme_paths, desc='NOT_MEME'):
    notmeme_texts[str(p)] = extract_ocr_text(p, ocr_reader)

print(f'\n텍스트 감지율:')
print(f'  MEME:     {sum(1 for t in meme_texts.values() if t)}/{len(meme_paths)}')
print(f'  NOT_MEME: {sum(1 for t in notmeme_texts.values() if t)}/{len(notmeme_paths)}')

# ── OCR 완료 후 메모리 정리 ──────────────────────────────────
del ocr_reader
import gc; gc.collect()
torch.cuda.empty_cache()
print(f'OCR 완료 후 VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated')


EasyOCR 초기화 중...


✅ 완료

MEME 502개 처리 중 (OCR + Visual Part)...


MEME:   0%|          | 0/502 [00:00<?, ?it/s]


NOT_MEME 736개 OCR 중...


NOT_MEME:   0%|          | 0/736 [00:00<?, ?it/s]


텍스트 감지율:
  MEME:     469/502
  NOT_MEME: 407/736
OCR 완료 후 VRAM: 0.0 GB allocated


In [7]:
# ── CLIP ViT-L/14@336 설정 + Dataset ──────────────────────────
# v3: ViT-L/14 (224px) → v4: ViT-L/14@336px
# 336px는 CLIP 공식 고해상도 버전 (동일 아키텍처, 더 세밀한 패치)
# 밈 텍스트 같은 세부 특징 포착에 유리

CLIP_NAME = 'openai/clip-vit-large-patch14-336'
processor = CLIPProcessor.from_pretrained(CLIP_NAME)
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE     = torch.bfloat16  # A100 네이티브 bfloat16: float16보다 수치 안정적
IMG_SIZE  = 336

print(f'CLIP: {CLIP_NAME}')
print(f'입력 해상도: {IMG_SIZE}px  (v3: 224px)')
print(f'연산 정밀도: {DTYPE}  (A100 최적)')

# PIL 증강 (CLIP processor 적용 전)
PIL_TRAIN = T.Compose([
    T.Resize((int(IMG_SIZE*1.14), int(IMG_SIZE*1.14))),  # 382x382
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandAugment(num_ops=2, magnitude=8),
])
PIL_VAL    = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE))])
RAND_ERASE = T.RandomErasing(p=0.25, scale=(0.02, 0.15))  # 텐서 레벨 (processor 후 적용)

# TTA 12-view (v3: 10-view → v4: 12-view)
S = IMG_SIZE
TTA_AUGS = [
    T.Compose([T.Resize((S, S))]),
    T.Compose([T.Resize((int(S*1.14), int(S*1.14))), T.CenterCrop(S)]),
    T.Compose([T.Resize((S, S)), T.RandomHorizontalFlip(p=1.0)]),
    T.Compose([T.Resize((int(S*1.14), int(S*1.14))), T.FiveCrop(S), T.Lambda(lambda c: c[0])]),
    T.Compose([T.Resize((int(S*1.14), int(S*1.14))), T.FiveCrop(S), T.Lambda(lambda c: c[1])]),
    T.Compose([T.Resize((int(S*1.14), int(S*1.14))), T.FiveCrop(S), T.Lambda(lambda c: c[2])]),
    T.Compose([T.Resize((int(S*1.14), int(S*1.14))), T.FiveCrop(S), T.Lambda(lambda c: c[3])]),
    T.Compose([T.Resize((int(S*1.14), int(S*1.14))), T.FiveCrop(S), T.Lambda(lambda c: c[4])]),
    T.Compose([T.Resize((int(S*1.3), int(S*1.3))), T.CenterCrop(S)]),
    T.Compose([T.Resize((int(S*1.3), int(S*1.3))), T.CenterCrop(S), T.RandomHorizontalFlip(p=1.0)]),
    T.Compose([T.Resize((int(S*1.07), int(S*1.07))), T.CenterCrop(S)]),
    T.Compose([T.Resize((int(S*1.07), int(S*1.07))), T.CenterCrop(S), T.RandomHorizontalFlip(p=1.0)]),
]

DEFAULT_TEXT = 'an image'

class MemeDatasetCLIP(Dataset):
    def __init__(self, paths, labels, texts, is_train=False):
        self.paths    = [str(p) for p in paths]
        self.labels   = labels
        self.texts    = texts
        self.is_train = is_train

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img  = Image.open(self.paths[idx]).convert('RGB')
        img  = PIL_TRAIN(img) if self.is_train else PIL_VAL(img)
        text = self.texts[idx] if self.texts[idx].strip() else DEFAULT_TEXT
        enc  = processor(images=img, text=text, return_tensors='pt',
                          padding='max_length', max_length=77, truncation=True)
        pv   = enc['pixel_values'].squeeze(0)
        if self.is_train:
            pv = RAND_ERASE(pv)  # RandomErasing: 텍스트 영역 가리기 효과
        return {
            'pixel_values' : pv,
            'input_ids'    : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label'        : torch.tensor(self.labels[idx], dtype=torch.long)
        }

print('Dataset 클래스 정의 완료')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP: openai/clip-vit-large-patch14-336
입력 해상도: 336px  (v3: 224px)
연산 정밀도: torch.bfloat16  (A100 최적)
Dataset 클래스 정의 완료


In [8]:
# ── 모델 정의: Gated Fusion + 강화된 분류기 ──────────────────
# v3: concat(img, txt, cos_sim) → MLP  (1537차원 고정 융합)
# v4: GatedFusion(img, txt) = gate * img + (1-gate) * txt
#   → 밈 텍스트가 강할 때 자동으로 텍스트 특징에 더 집중
#   → concat(img, txt, gated, cos_sim) → MLP  (2305차원)

class GatedFusion(nn.Module):
    """학습 가능한 게이트로 이미지/텍스트 특징을 적응적으로 융합."""
    def __init__(self, dim: int):
        super().__init__()
        self.gate_net = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Linear(dim, dim),
            nn.Sigmoid()
        )
        self.ln_i = nn.LayerNorm(dim)
        self.ln_t = nn.LayerNorm(dim)

    def forward(self, img_feat, txt_feat):
        gate  = self.gate_net(torch.cat([img_feat, txt_feat], dim=-1))  # [B, dim]
        fused = gate * self.ln_i(img_feat) + (1 - gate) * self.ln_t(txt_feat)
        return fused, gate  # gate 반환 (모니터링/해석용)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma; self.alpha = alpha; self.ls = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha,
                              label_smoothing=self.ls, reduction='none')
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()


class MemeDetectorV4(nn.Module):
    def __init__(self, clip_name: str, dropout: float = 0.4):
        super().__init__()
        self.clip   = CLIPModel.from_pretrained(clip_name)
        proj_dim    = self.clip.config.projection_dim  # 768 for L/14
        self.fusion = GatedFusion(proj_dim)

        # 입력: img(768) + txt(768) + gated(768) + cos_sim(1) = 2305
        in_dim = proj_dim * 3 + 1
        self.classifier = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 768),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, 2)
        )
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                nn.init.zeros_(m.bias)

    def encode_image(self, pixel_values):
        out = self.clip.vision_model(pixel_values=pixel_values, return_dict=True)
        return self.clip.visual_projection(out.pooler_output)  # [B, 768]

    def encode_text(self, input_ids, attention_mask):
        out = self.clip.text_model(input_ids=input_ids,
                                    attention_mask=attention_mask, return_dict=True)
        return self.clip.text_projection(out.pooler_output)    # [B, 768]

    def forward(self, pixel_values, input_ids, attention_mask):
        img = F.normalize(self.encode_image(pixel_values), dim=-1)
        txt = F.normalize(self.encode_text(input_ids, attention_mask), dim=-1)
        cos = (img * txt).sum(dim=-1, keepdim=True)             # [B, 1]
        fused, gate = self.fusion(img, txt)                     # [B, 768]
        combined = torch.cat([img, txt, fused, cos], dim=-1)    # [B, 2305]
        return self.classifier(combined)

# 테스트 실행
print('모델 초기화 테스트...')
_m = MemeDetectorV4(CLIP_NAME).to(device)
_e = processor(images=Image.new('RGB', (336,336)), text='meme test',
               return_tensors='pt', padding='max_length', max_length=77, truncation=True)
with torch.no_grad(), autocast('cuda', dtype=DTYPE):
    _o = _m(_e['pixel_values'].to(device), _e['input_ids'].to(device), _e['attention_mask'].to(device))
print(f'✅ 출력: {tuple(_o.shape)}  (expected: (1, 2))')
print(f'   분류기 파라미터: {sum(p.numel() for p in _m.classifier.parameters()):,}')
print(f'   전체 파라미터  : {sum(p.numel() for p in _m.parameters()):,}')
del _m; torch.cuda.empty_cache()

모델 초기화 테스트...


pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14-336
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

✅ 출력: (1, 2)  (expected: (1, 2))
   분류기 파라미터: 1,972,996
   전체 파라미터  : 431,692,805


In [9]:
# ── 학습 유틸리티 (CutMix, MixUp, LLRD, 스케줄러) ────────────

def mixup_data(pv, lbl, alpha=0.2):
    """MixUp: 두 이미지를 선형 보간."""
    if alpha <= 0: return pv, lbl, lbl, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(pv.size(0), device=pv.device)
    return lam*pv + (1-lam)*pv[idx], lbl, lbl[idx], lam

def cutmix_data(pv, lbl, alpha=1.0):
    """CutMix: 직사각형 패치를 다른 이미지로 교체.
    MixUp보다 더 강한 정규화 효과 (자연스러운 경계 유지).
    [수정] pv.clone() 제거 → 배치 2배 메모리 점유 문제 해결."""
    if alpha <= 0: return pv, lbl, lbl, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(pv.size(0), device=pv.device)
    B, C, H, W = pv.shape
    r  = np.sqrt(1 - lam)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bw, bh = int(W*r), int(H*r)
    x1, x2 = max(0, cx-bw//2), min(W, cx+bw//2)
    y1, y2 = max(0, cy-bh//2), min(H, cy+bh//2)
    lam_real = 1 - (x2-x1)*(y2-y1)/(W*H)
    # clone() 없이 슬라이싱으로 직접 조합
    rows = []
    if y1 > 0:   rows.append(pv[:, :, :y1, :])
    mid = torch.cat([
        pv[:, :, y1:y2, :x1],
        pv[idx, :, y1:y2, x1:x2],
        pv[:, :, y1:y2, x2:]
    ], dim=3) if (x2 > x1) else pv[:, :, y1:y2, :]
    rows.append(mid)
    if y2 < H:   rows.append(pv[:, :, y2:, :])
    pv_mix = torch.cat(rows, dim=2) if len(rows) > 1 else rows[0]
    return pv_mix, lbl, lbl[idx], lam_real

def get_llrd_groups(model, base_lr, decay=0.88):
    """LLRD: 상위 레이어일수록 높은 LR (사전학습 보존)."""
    no_wd = ['bias', 'LayerNorm.weight']
    head_nps   = [(n, p) for n, p in model.named_parameters()
                  if 'classifier' in n or 'fusion' in n]
    head_names = {n for n, _ in head_nps}
    vis, txt, other = {}, {}, []
    for n, p in model.named_parameters():
        if n in head_names: continue
        if 'vision_model.encoder.layers.' in n:
            i = int(n.split('vision_model.encoder.layers.')[1].split('.')[0])
            vis.setdefault(i, []).append((n, p))
        elif 'text_model.encoder.layers.' in n:
            i = int(n.split('text_model.encoder.layers.')[1].split('.')[0])
            txt.setdefault(i, []).append((n, p))
        else:
            other.append((n, p))
    groups = [{'params': [p for _, p in head_nps],
                'lr': base_lr * 10, 'weight_decay': 0.01}]
    for layers in [vis, txt]:
        nmax = max(layers.keys())+1 if layers else 0
        for i in sorted(layers.keys(), reverse=True):
            lr_i = base_lr * (decay ** (nmax - i))
            nps  = layers[i]
            groups += [
                {'params': [p for n,p in nps if not any(nd in n for nd in no_wd)],
                 'lr': lr_i, 'weight_decay': 0.01},
                {'params': [p for n,p in nps if any(nd in n for nd in no_wd)],
                 'lr': lr_i, 'weight_decay': 0.0},
            ]
    nmax = max(max(vis.keys(), default=0), max(txt.keys(), default=0))
    emb_lr = base_lr * (decay ** (nmax + 1))
    groups += [
        {'params': [p for n,p in other if not any(nd in n for nd in no_wd)],
         'lr': emb_lr, 'weight_decay': 0.01},
        {'params': [p for n,p in other if any(nd in n for nd in no_wd)],
         'lr': emb_lr, 'weight_decay': 0.0},
    ]
    return [g for g in groups if g['params']]

def cosine_warmup(optimizer, warmup_steps, total_steps):
    def fn(step):
        if step < warmup_steps: return step / max(1, warmup_steps)
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * p)))
    return LambdaLR(optimizer, fn)

print('학습 유틸리티 정의 완료')

학습 유틸리티 정의 완료


In [ ]:
import os, gc
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'  # 메모리 단편화 방지

# ── 5-Fold CV 학습 ─────────────────────────────────────────────
# 핵심 전략:
#   전체 데이터 → test 15% 고정 분리
#   나머지 85% → 5-Fold 교차검증
#   각 fold에서 최고 모델 저장 → test에서 5모델 앙상블
#
# 효과: 단일 split 대비 편향↓, 분산↓, 과적합↓

# ── 하이퍼파라미터 ─────────────────────────────────────────────
PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 30
TOTAL_EPOCHS  = PHASE1_EPOCHS + PHASE2_EPOCHS
HEAD_LR       = 1e-3
BASE_LR       = 4e-6     # L/14@336은 L/14보다 약간 낮게
LLRD_DECAY    = 0.88
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
MAX_GRAD_NORM = 1.0
PATIENCE      = 8
MIXUP_ALPHA   = 0.2
CUTMIX_ALPHA  = 1.0
CUTMIX_PROB   = 0.5      # 50% CutMix, 50% MixUp
N_FOLDS       = 5
BATCH_SIZE    = 64       # A100: 32 → 64

# ── Train/Test 분리 ──────────────────────────────────────────
all_paths  = meme_paths + notmeme_paths
all_labels = [1]*len(meme_paths) + [0]*len(notmeme_paths)
all_texts  = ([meme_texts.get(str(p),'') for p in meme_paths] +
               [notmeme_texts.get(str(p),'') for p in notmeme_paths])

kf_paths, test_paths, kf_labels, test_labels, kf_texts, test_texts = train_test_split(
    all_paths, all_labels, all_texts,
    test_size=0.15, stratify=all_labels, random_state=SEED
)
print(f'KFold pool: {len(kf_paths)}개  |  Test: {len(test_paths)}개')

# ── 5-Fold 학습 루프 ─────────────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_ckpts    = []
fold_histories = []
fold_best_accs = []

for fold_idx, (tr_idx, vl_idx) in enumerate(skf.split(kf_paths, kf_labels)):
    print(f'\n{"="*60}')
    print(f'  FOLD {fold_idx+1}/{N_FOLDS}')
    print(f'{"="*60}')

    # 데이터 준비
    tr_paths_b = [kf_paths[i] for i in tr_idx]
    tr_lbl_b   = [kf_labels[i] for i in tr_idx]
    tr_txt_b   = [kf_texts[i]  for i in tr_idx]
    vl_paths   = [kf_paths[i] for i in vl_idx]
    vl_labels  = [kf_labels[i] for i in vl_idx]
    vl_texts   = [kf_texts[i]  for i in vl_idx]

    # Visual Part hard negatives → train에만 추가
    tr_meme  = [p for p, l in zip(tr_paths_b, tr_lbl_b) if l == 1]
    vp_ps    = [vp_map[str(p)] for p in tr_meme if str(p) in vp_map]
    tr_paths = tr_paths_b + vp_ps
    tr_lbl   = tr_lbl_b + [0]*len(vp_ps)
    tr_txt   = tr_txt_b + ['']*len(vp_ps)

    lbl_cnt   = Counter(tr_lbl)
    total_tr  = len(tr_lbl)
    cls_w     = {c: total_tr/cnt for c, cnt in lbl_cnt.items()}
    cw_tensor = torch.tensor([cls_w[0], cls_w[1]], dtype=torch.float).to(device)

    train_ds = MemeDatasetCLIP(tr_paths, tr_lbl,   tr_txt,   is_train=True)
    val_ds   = MemeDatasetCLIP(vl_paths, vl_labels, vl_texts, is_train=False)

    sw       = [cls_w[l] for l in tr_lbl]
    sampler  = WeightedRandomSampler(sw, len(tr_lbl), replacement=True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                           num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)

    print(f'  Train: {len(train_ds)}개  Val: {len(val_ds)}개  '
          f'(VP: {len(vp_ps)}개 포함)')

    # 모델 초기화
    model     = MemeDetectorV4(CLIP_NAME).to(device)
    criterion = FocalLoss(gamma=2.0, alpha=cw_tensor, label_smoothing=0.05)
    best_val  = 0.0
    patience  = 0
    history   = []
    ckpt_path = f'{OUT_DIR}/fold{fold_idx+1}_best.pth'

    # ── Phase 1: Head만 학습 ────────────────────────────────
    print(f'  [Phase 1] Classifier+Fusion 학습 ({PHASE1_EPOCHS} ep)')
    for n, p in model.named_parameters():
        p.requires_grad = ('classifier' in n or 'fusion' in n)

    head_ps = [p for n, p in model.named_parameters() if p.requires_grad]
    opt1    = AdamW(head_ps, lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    sch1    = cosine_warmup(opt1,
                warmup_steps=int(PHASE1_EPOCHS*len(train_dl)*0.2),
                total_steps=PHASE1_EPOCHS*len(train_dl))

    for ep in range(1, PHASE1_EPOCHS+1):
        model.train()
        tc, tt = 0, 0
        for batch in tqdm(train_dl, desc=f'  Ep{ep}[P1]', leave=False):
            pv  = batch['pixel_values'].to(device)
            iid = batch['input_ids'].to(device)
            am  = batch['attention_mask'].to(device)
            lbl = batch['label'].to(device)
            opt1.zero_grad(set_to_none=True)
            with autocast('cuda', dtype=DTYPE):  # bfloat16: GradScaler 불필요
                logits = model(pv, iid, am)
                loss   = criterion(logits, lbl)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            opt1.step(); sch1.step()
            tc += (logits.argmax(1) == lbl).sum().item(); tt += lbl.size(0)

        model.eval(); vc, vt = 0, 0
        with torch.no_grad():
            for b in val_dl:
                with autocast('cuda', dtype=DTYPE):
                    lo = model(b['pixel_values'].to(device),
                               b['input_ids'].to(device),
                               b['attention_mask'].to(device))
                vc += (lo.argmax(1) == b['label'].to(device)).sum().item()
                vt += b['label'].size(0)
        va = vc/vt
        print(f'  Ep{ep}/{TOTAL_EPOCHS} [P1] train={tc/tt:.4f} val={va:.4f}')
        history.append({'epoch': ep, 'val_acc': va, 'phase': 'HEAD'})
        if va > best_val:
            best_val = va; patience = 0
            torch.save(model.state_dict(), ckpt_path)

    # ── Phase 1 마무리: 옵티마이저 해제 ──────────────────────
    del opt1, sch1
    torch.cuda.empty_cache(); gc.collect()

    # ── Phase 2: 전체 LLRD + CutMix/MixUp ──────────────────
    print(f'  [Phase 2] Full fine-tuning + CutMix/MixUp ({PHASE2_EPOCHS} ep)')
    for p in model.parameters(): p.requires_grad = True

    opt2   = AdamW(get_llrd_groups(model, BASE_LR, LLRD_DECAY))
    p2_st  = PHASE2_EPOCHS * len(train_dl)
    sch2   = cosine_warmup(opt2,
                warmup_steps=int(p2_st * WARMUP_RATIO),
                total_steps=p2_st)
    patience = 0

    for ep in range(PHASE1_EPOCHS+1, TOTAL_EPOCHS+1):
        model.train(); tc, tt = 0, 0
        for batch in tqdm(train_dl, desc=f'  Ep{ep}[P2]', leave=False):
            pv  = batch['pixel_values'].to(device)
            iid = batch['input_ids'].to(device)
            am  = batch['attention_mask'].to(device)
            lbl = batch['label'].to(device)
            # CutMix / MixUp 랜덤 선택
            if np.random.rand() < CUTMIX_PROB:
                pv, la, lb, lam = cutmix_data(pv, lbl, CUTMIX_ALPHA)
            else:
                pv, la, lb, lam = mixup_data(pv, lbl, MIXUP_ALPHA)
            opt2.zero_grad(set_to_none=True)
            with autocast('cuda', dtype=DTYPE):
                logits = model(pv, iid, am)
                loss = (lam*criterion(logits, la) + (1-lam)*criterion(logits, lb)
                        if lam < 1.0 else criterion(logits, lbl))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            opt2.step(); sch2.step()
            tc += (logits.argmax(1) == lbl).sum().item(); tt += lbl.size(0)
            del pv, iid, am, lbl, logits, loss
            if tt % (BATCH_SIZE * 10) == 0:
                torch.cuda.empty_cache()

        model.eval(); vc, vt = 0, 0
        with torch.no_grad():
            for b in val_dl:
                with autocast('cuda', dtype=DTYPE):
                    lo = model(b['pixel_values'].to(device),
                               b['input_ids'].to(device),
                               b['attention_mask'].to(device))
                vc += (lo.argmax(1) == b['label'].to(device)).sum().item()
                vt += b['label'].size(0)
        va = vc/vt
        print(f'  Ep{ep}/{TOTAL_EPOCHS} [P2] train={tc/tt:.4f} val={va:.4f}')
        history.append({'epoch': ep, 'val_acc': va, 'phase': 'FULL'})
        if va > best_val:
            best_val = va; patience = 0
            torch.save(model.state_dict(), ckpt_path)
            print(f'  ★ Best: fold{fold_idx+1} val={va:.4f}')
        else:
            patience += 1
            if patience >= PATIENCE:
                print(f'  ■ Early stopping'); break

    print(f'  Fold {fold_idx+1} 완료: Best Val={best_val:.4f}')
    fold_ckpts.append(ckpt_path)
    fold_histories.append(history)
    fold_best_accs.append(best_val)

    # ── 폴드 종료: 반드시 메모리 해제 (예외 발생 시에도) ──
    try:
        del opt2, sch2
    except NameError:
        pass
    del model
    torch.cuda.empty_cache()
    gc.collect()
    print(f'  VRAM 해제 후: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated')

print(f'\n✅ 5-Fold 학습 완료')
print(f'   Fold별 Best Val: {[f"{a:.4f}" for a in fold_best_accs]}')
print(f'   평균: {np.mean(fold_best_accs):.4f}  std: {np.std(fold_best_accs):.4f}')

KFold pool: 1052개  |  Test: 186개

  FOLD 1/5
  Train: 1182개  Val: 211개  (VP: 341개 포함)


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14-336
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [Phase 1] Classifier+Fusion 학습 (5 ep)


  Ep1[P1]:   0%|          | 0/19 [00:00<?, ?it/s]

...

  Ep13/35 [P2] train=0.8488 val=0.8714


  Ep14[P2]:   0%|          | 0/19 [00:00<?, ?it/s]

  Ep14/35 [P2] train=0.8345 val=0.8714


  Ep15[P2]:   0%|          | 0/19 [00:00<?, ?it/s]

  Ep15/35 [P2] train=0.8302 val=0.8857
  ■ Early stopping
  Fold 5 완료: Best Val=0.8905
  VRAM 해제 후: 0.0 GB allocated

✅ 5-Fold 학습 완료
   Fold별 Best Val: ['0.9005', '0.9289', '0.8667', '0.9000', '0.8905']
   평균: 0.8973  std: 0.0200


In [ ]:
# ── TTA 12-view × 5-Fold 앙상블 평가 ──────────────────────────
# 총 60회 추론(5 모델 × 12 뷰)의 softmax 평균
print(f'앙상블 평가: {N_FOLDS}개 모델 × {len(TTA_AUGS)}-view TTA = {N_FOLDS*len(TTA_AUGS)}회 추론')

def predict_ensemble_tta(paths, texts, tta_augs, ckpt_paths, batch_size=64):
    all_probs = []
    for ckpt in ckpt_paths:
        print(f'  로드: {Path(ckpt).name}')
        m = MemeDetectorV4(CLIP_NAME).to(device)
        m.load_state_dict(torch.load(ckpt, map_location=device))
        m.eval()
        view_probs = []
        for aug in tqdm(tta_augs, desc='  TTA', leave=False):
            bufs = {'pv': [], 'iid': [], 'am': []}
            vp   = []
            def flush():
                if not bufs['pv']: return
                pv_  = torch.stack(bufs['pv']).to(device)
                iid_ = torch.stack(bufs['iid']).to(device)
                am_  = torch.stack(bufs['am']).to(device)
                with torch.no_grad(), autocast('cuda', dtype=DTYPE):
                    lo = m(pv_, iid_, am_)
                    vp.append(torch.softmax(lo.float(), dim=1).cpu().numpy())
                del pv_, iid_, am_, lo  # GPU 텐서 즉시 해제
                bufs['pv'].clear(); bufs['iid'].clear(); bufs['am'].clear()
            for i, (p, t) in enumerate(zip(paths, texts)):
                img  = aug(Image.open(str(p)).convert('RGB'))
                text = t if t.strip() else DEFAULT_TEXT
                enc  = processor(images=img, text=text, return_tensors='pt',
                                  padding='max_length', max_length=77, truncation=True)
                bufs['pv'].append(enc['pixel_values'].squeeze(0))
                bufs['iid'].append(enc['input_ids'].squeeze(0))
                bufs['am'].append(enc['attention_mask'].squeeze(0))
                if len(bufs['pv']) == batch_size or i == len(paths)-1:
                    flush()
            view_probs.append(np.concatenate(vp, axis=0))
        all_probs.append(np.mean(view_probs, axis=0))
        del m; torch.cuda.empty_cache()
    avg  = np.mean(all_probs, axis=0)
    return avg.argmax(axis=1), avg[:, 1]

test_preds, test_scores = predict_ensemble_tta(
    test_paths, test_texts, TTA_AUGS, fold_ckpts, batch_size=64
)
test_true = np.array(test_labels)

acc = accuracy_score(test_true, test_preds)
f1  = f1_score(test_true, test_preds, zero_division=0)
auc = roc_auc_score(test_true, test_scores)
cm  = confusion_matrix(test_true, test_preds)

print()
print('=' * 65)
print('  MemeTector v4 — L/14@336 + 5-Fold Ensemble + TTA 12-view')
print('-' * 65)
print(f'  [Test]  Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')
print(f'  v3 대비  Acc: 0.8600 → {acc:.4f} ({(acc-0.86)*100:+.1f}%)')
print(f'          AUC: 0.9426 → {auc:.4f} ({(auc-0.9426)*100:+.2f}%)')
print('-' * 65)
print(f'  Confusion Matrix:')
print(f'             NOT_MEME  MEME')
print(f'  NOT_MEME   {cm[0][0]:>6}   {cm[0][1]:>6}')
print(f'  MEME       {cm[1][0]:>6}   {cm[1][1]:>6}')
print('=' * 65)
print()
print(classification_report(test_true, test_preds,
      target_names=['NOT_MEME','MEME'], zero_division=0))

In [ ]:
# ── 결과 시각화 + 저장 ─────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
colors = plt.cm.tab10(np.linspace(0, 1, N_FOLDS))

# 1. Fold별 학습 곡선
ax = axes[0, 0]
for fi, hist in enumerate(fold_histories):
    eps  = [h['epoch'] for h in hist]
    vaccs = [h['val_acc'] for h in hist]
    ax.plot(eps, vaccs, color=colors[fi], label=f'Fold {fi+1}', alpha=0.8, linewidth=1.5)
ax.axhline(np.mean(fold_best_accs), linestyle='--', color='red',
           label=f'Mean Best={np.mean(fold_best_accs):.3f}', linewidth=2)
ax.set(xlabel='Epoch', ylabel='Val Accuracy', title='5-Fold 학습 곡선')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 2. Fold별 Best Val Acc
ax = axes[0, 1]
bars = ax.bar(range(1, N_FOLDS+1), fold_best_accs, color=colors, alpha=0.85)
ax.axhline(np.mean(fold_best_accs), linestyle='--', color='red',
           label=f'Mean={np.mean(fold_best_accs):.3f}')
ax.set(xlabel='Fold', ylabel='Best Val Acc', title='Fold별 최고 Val Accuracy', ylim=[0.8, 1.0])
ax.legend(); ax.grid(alpha=0.3, axis='y')
for bar, a in zip(bars, fold_best_accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{a:.3f}', ha='center', va='bottom', fontsize=9)

# 3. Confusion Matrix
ax = axes[0, 2]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['NOT_MEME','MEME'], yticklabels=['NOT_MEME','MEME'],
            annot_kws={'size': 14})
ax.set(xlabel='Predicted', ylabel='True',
       title=f'Confusion Matrix (Test)\nAcc={acc:.3f}  F1={f1:.3f}')

# 4. ROC Curve
ax = axes[1, 0]
fpr, tpr, _ = roc_curve(test_true, test_scores)
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'v4 AUC={auc:.4f}')
ax.plot([0,1],[0,1], '--', color='navy', lw=1, label='v3 AUC=0.9426 (참고)')
ax.set(xlim=[0,1], ylim=[0,1.02], xlabel='FPR', ylabel='TPR', title='ROC Curve')
ax.legend(); ax.grid(alpha=0.3)

# 5. v3 vs v4 성능 비교
ax = axes[1, 1]
metrics  = ['Accuracy', 'F1-Score', 'AUC']
v3_vals  = [0.8600, 0.8696, 0.9426]
v4_vals  = [acc, f1, auc]
x = np.arange(len(metrics)); bw = 0.35
ax.bar(x - bw/2, v3_vals, bw, label='v3 (기준)', color='steelblue', alpha=0.8)
ax.bar(x + bw/2, v4_vals, bw, label='v4 (개선)', color='tomato',    alpha=0.8)
for i, (v3, v4) in enumerate(zip(v3_vals, v4_vals)):
    ax.text(i+bw/2, v4+0.005, f'{(v4-v3)*100:+.1f}%',
            ha='center', va='bottom', color='red', fontsize=9, fontweight='bold')
ax.set(xticks=x, xticklabels=metrics, ylim=[0.8, 1.0],
       ylabel='Score', title='v3 vs v4 성능 비교')
ax.legend(); ax.grid(alpha=0.3, axis='y')

# 6. 예측 신뢰도 분포
ax = axes[1, 2]
ax.hist(test_scores[test_true==0], bins=20, alpha=0.6, color='steelblue',
        label='NOT_MEME', density=True)
ax.hist(test_scores[test_true==1], bins=20, alpha=0.6, color='tomato',
        label='MEME', density=True)
ax.axvline(0.5, linestyle='--', color='black', label='Threshold=0.5')
ax.set(xlabel='MEME 예측 확률', ylabel='밀도', title='예측 신뢰도 분포')
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle(
    f'MemeTector v4  |  CLIP L/14@336 + 5-Fold + TTA 12-view\n'
    f'Test: Acc={acc*100:.1f}%  F1={f1:.3f}  AUC={auc:.4f}  '
    f'(v3 대비 Acc {(acc-0.86)*100:+.1f}%)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/results_v4.png', dpi=150, bbox_inches='tight')
plt.show()

# 요약 저장
summary = {
    'model': 'CLIP ViT-L/14@336 Multimodal + Gated Fusion',
    'v4_improvements': [
        'CLIP ViT-L/14@336px (224→336 고해상도)',
        '전체 데이터 사용 (502 MEME, 736 NOT_MEME)',
        f'5-Fold CV 앙상블 (평균 val={np.mean(fold_best_accs):.4f})',
        'Gated Fusion (학습 가능한 이미지/텍스트 가중합)',
        'CutMix + MixUp 혼합 (50/50)',
        'RandomErasing (p=0.25)',
        'bfloat16 (A100 최적화)',
        'Batch 64',
        'TTA 12-view',
    ],
    'n_folds': N_FOLDS,
    'fold_best_val_accs': [round(a, 4) for a in fold_best_accs],
    'fold_mean_val_acc':  round(float(np.mean(fold_best_accs)), 4),
    'test_accuracy': round(float(acc), 4),
    'test_f1':       round(float(f1),  4),
    'test_roc_auc':  round(float(auc), 4),
    'confusion_matrix': cm.tolist(),
    'v3_test_accuracy': 0.8600,
    'v3_test_auc':      0.9426,
    'improvement_acc':  round(float(acc) - 0.8600, 4),
    'improvement_auc':  round(float(auc) - 0.9426, 4),
}
with open(f'{OUT_DIR}/summary_v4.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'\n✅ 저장 완료: {OUT_DIR}/')
print(f'   fold1~5_best.pth  |  results_v4.png  |  summary_v4.json')
print(f'\n=== 최종 요약 ===')
print(f'  v3 Test Acc: 86.0%  →  v4 Test Acc: {acc*100:.1f}%  ({(acc-0.86)*100:+.1f}%)')
print(f'  v3 AUC:      0.9426 →  v4 AUC:      {auc:.4f}  ({(auc-0.9426)*100:+.2f}%)')